# Notebook 01: Dataset Verification and Standardization

## Objective

This notebook performs the initial verification and standardization of all cleaned datasets used in the UniMTScore framework.

The objectives are:

- Load all cleaned datasets.
- Verify dataset integrity.
- Check for missing and duplicate values.
- Inspect dataset structure and column names.
- Standardize column names into a common schema.
- Save standardized datasets for subsequent evaluation modules.

### Datasets Used

1. FLORES-200
2. IndicMT-Eval
3. IN22 Benchmark

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', 'Fruit_dataset.zip', 'Bajaj', 'UniMTScore']


In [4]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

FLORES_DIR = PROJECT_ROOT / "datasets" / "FLORES200"
INDICMT_DIR = PROJECT_ROOT / "datasets" / "IndicMT_Eval"
IN22_DIR = PROJECT_ROOT / "datasets" / "IN22"

print(list(FLORES_DIR.glob("*")))

[PosixPath('/content/drive/MyDrive/UniMTScore/datasets/FLORES200/Gujarati_clean.xlsx'), PosixPath('/content/drive/MyDrive/UniMTScore/datasets/FLORES200/Hindi_clean.xlsx'), PosixPath('/content/drive/MyDrive/UniMTScore/datasets/FLORES200/Kannada_clean.xlsx'), PosixPath('/content/drive/MyDrive/UniMTScore/datasets/FLORES200/Malayalam_clean.xlsx'), PosixPath('/content/drive/MyDrive/UniMTScore/datasets/FLORES200/Marathi_clean.xlsx'), PosixPath('/content/drive/MyDrive/UniMTScore/datasets/FLORES200/Odia_clean.xlsx')]


In [5]:
flores_files = list(FLORES_DIR.glob("*"))

print(flores_files)

[PosixPath('/content/drive/MyDrive/UniMTScore/datasets/FLORES200/Gujarati_clean.xlsx'), PosixPath('/content/drive/MyDrive/UniMTScore/datasets/FLORES200/Hindi_clean.xlsx'), PosixPath('/content/drive/MyDrive/UniMTScore/datasets/FLORES200/Kannada_clean.xlsx'), PosixPath('/content/drive/MyDrive/UniMTScore/datasets/FLORES200/Malayalam_clean.xlsx'), PosixPath('/content/drive/MyDrive/UniMTScore/datasets/FLORES200/Marathi_clean.xlsx'), PosixPath('/content/drive/MyDrive/UniMTScore/datasets/FLORES200/Odia_clean.xlsx')]


In [ ]:
import pandas as pd

flores_data = {}

for file in FLORES_DIR.glob("*.xlsx"):
    language = file.stem.replace("_clean", "")
    df = pd.read_excel(file)
    flores_data[language] = df

print(f"Loaded {len(flores_data)} FLORES datasets.")
print("Languages:", list(flores_data.keys()))

Loaded 0 FLORES datasets.
Languages: []


In [ ]:
for language, df in flores_data.items():
    print("=" * 60)
    print(language)
    print(df.columns.tolist())

In [ ]:
def verify_dataset(df, name):

    summary = {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": int(df.isnull().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum())
    }

    print("=" * 70)
    print(name)
    print("=" * 70)

    print(f"Rows           : {summary['Rows']}")
    print(f"Columns        : {summary['Columns']}")
    print(f"Missing Values : {summary['Missing Values']}")
    print(f"Duplicate Rows : {summary['Duplicate Rows']}")

    print("\nColumn Names")
    print(df.columns.tolist())

    print("\nData Types")
    print(df.dtypes)

    print("\nSample Data")
    display(df.head())

    return summary

In [ ]:
verification_results = []

for language, df in flores_data.items():
    summary = verify_dataset(df, language)
    verification_results.append(summary)

In [ ]:
verification_df = pd.DataFrame(verification_results)

verification_df

""


In [ ]:
def standardize_flores(df, language):

    # Find the non-English column
    target_column = [col for col in df.columns if col != "sentence_eng_Latn"][0]

    standardized = pd.DataFrame({
        "source": df["sentence_eng_Latn"],
        "reference": df[target_column],
        "language": language,
        "dataset": "FLORES200"
    })

    return standardized

In [ ]:
standardized_flores = {}

for language, df in flores_data.items():

    standardized_df = standardize_flores(df, language)

    standardized_flores[language] = standardized_df

In [ ]:
standardized_flores["Hindi"].head()

KeyError: 'Hindi'

In [ ]:
STANDARDIZED_DIR = PROJECT_ROOT / "processed" / "flores"
STANDARDIZED_DIR.mkdir(parents=True, exist_ok=True)

for language, df in standardized_flores.items():

    output_file = STANDARDIZED_DIR / f"{language}_standardized.csv"

    df.to_csv(output_file, index=False)

print("✅ All standardized FLORES datasets have been saved.")

✅ All standardized FLORES datasets have been saved.


In [ ]:
verification_report = []

for language, df in standardized_flores.items():

    verification_report.append({
        "Language": language,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": int(df.isnull().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Status": "Verified"
    })

verification_report = pd.DataFrame(verification_report)

verification_report

""


## FLORES-200 Dataset Standardization

The cleaned FLORES-200 datasets contained one English sentence column and one target language sentence column. Since the target language column name differed across languages, all datasets were transformed into a unified schema consisting of four columns:

- **source**: English source sentence
- **reference**: Human reference translation
- **language**: Target language identifier
- **dataset**: Dataset source (FLORES200)

This standardization enables the subsequent lexical, syntactic, and semantic evaluation modules to operate on a common data format regardless of the language.

In [ ]:
train = pd.read_csv(INDICMT_DIR / "train_clean.csv")
validation = pd.read_csv(INDICMT_DIR / "validation_clean.csv")
test = pd.read_csv(INDICMT_DIR / "test_clean.csv")

print("Train Shape      :", train.shape)
print("Validation Shape :", validation.shape)
print("Test Shape       :", test.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/UniMTScore/datasets/IndicMT_Eval/train_clean.csv'

In [ ]:
verify_dataset(train, "IndicMT Train")
verify_dataset(validation, "IndicMT Validation")
verify_dataset(test, "IndicMT Test")

IndicMT Train
Rows           : 4997
Columns        : 6
Missing Values : 5
Duplicate Rows : 25

Column Names
['src', 'ref', 'translation', 'language', 'mqm_norm_score', 'da_norm_score']

Data Types
src                object
ref                object
translation        object
language           object
mqm_norm_score    float64
da_norm_score     float64
dtype: object

Sample Data


,src,ref,translation,language,mqm_norm_score,da_norm_score
0,NaN,NaN,NaN,Gujarati,NaN,NaN
1,. Scientists say this animal's plumage was chestnut-brown on top with a pale or carotenoid-colored underside.,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથીકથ્થાઈ-બદામી રંગના અને અંદરની બાજુથી આછા અથવા પીળા રંગના હતા.,એમ.વૈજ્ઞાનિકોનું કહેવું છે કે આ પ્રાણીના પીંછાં પીળા અથવા કેરોટિનોઇડ રંગના નીચેના ભાગ સાથે ટોચ પર ચેસ્ટનટ-બ્રાઉન રંગના હતા.,Gujarati,0.90,0.8
2,. Scientists say this animal's plumage was chestnut-brown on top with a pale or carotenoid-colored underside.,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથીકથ્થાઈ-બદામી રંગના અને અંદરની બાજુથી આછા અથવા પીળા રંગના હતા.,. વૈજ્entistsાનિકોકહે છે કે આ પ્રાણીનું પ્લgeમજ નિસ્તેજ અથવા કેરોટિનોઇડ રંગના અન્ડરસાઇડ સાથે ટોચ પર ચેસ્ટનટ-બ્રાઉન હતું.,Gujarati,0.60,0.6
3,. Scientists say this animal's plumage was chestnut-brown on top with a pale or carotenoid-colored underside.,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથીકથ્થાઈ-બદામી રંગના અને અંદરની બાજુથી આછા અથવા પીળા રંગના હતા.,. વૈજ્ઞાનિકોનું કહેવું છે કે આ પ્રાણીનું પ્લમેજ ટોચ પર ચેસ્ટનટ-બ્રાઉન હતું અને તેની નીચે ફિક્કો અથવા કેરોટીનોઇડ રંગનોહતો.,Gujarati,0.88,0.8
4,. Scientists say this animal's plumage was chestnut-brown on top with a pale orcarotenoid-colored underside.,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથીકથ્થાઈ-બદામી રંગના અને અંદરની બાજુથી આછા અથવા પીળા રંગના હતા.,વૈજ્ઞાનિકોનું કહેવું છે કે આ પ્રાણીની પલમેજ ઉપરના ભાગમાં ચૈસ્ટનટ-બ્રાઉન હતી અને તેની નીચેના ભાગમાં કેરોટેનોઈડનો રંગ હતો.,Gujarati,0.86,0.8


IndicMT Validation
Rows           : 1000
Columns        : 6
Missing Values : 0
Duplicate Rows : 7

Column Names
['src', 'ref', 'translation', 'language', 'mqm_norm_score', 'da_norm_score']

Data Types
src                object
ref                object
translation        object
language           object
mqm_norm_score    float64
da_norm_score     float64
dtype: object

Sample Data


,src,ref,translation,language,mqm_norm_score,da_norm_score
0,"The final match of the series will take place at Ellis Park in Johannesburg next week, when the Springboks play Australia.","या मालिकेतील अंतिम सामना हा जोहान्सबर्ग येथील एलीस पार्क मध्ये खेळवला जाईल, जेव्हा स्प्रिंगबॉक्स ऑस्ट्रेलियाशी खेळतील.","पुढील आठवड्यात जोहान्सबर्गच्या एलिस पार्कमध्ये या मालिकेचा अंतिम सामना होईल, जेव्हा स्प्रिंगबॉक्स ऑस्ट्रेलियाशीखेळेल.",Marathi,0.96,0.92
1,"The final match of the series will take place at Ellis Park in Johannesburg next week, when the Springboks play Australia.","या मालिकेतील अंतिम सामना हा जोहान्सबर्ग येथील एलीस पार्क मध्ये खेळवला जाईल, जेव्हा स्प्रिंगबॉक्स ऑस्ट्रेलियाशी खेळतील.","या मालिकेतील अंतिम सामना पुढील आठवड्यात जोहान्सबर्गमधील एलिस पार्क येथे होईल, जेव्हा स्प्रिंगबॉक्स ऑस्ट्रेलियाशी खेळेल.",Marathi,1.00,1.00
2,"The final score was a one-point victory, 21 to 20, ending the All Blacks' 15 game winning streak.","अखेरच्या धावांनी 21 ते 20 असा एक गुण राखून विजय मिळविला, ज्याने ऑल ब्लॅकचा 15 गेम जिंकणारा वेग समाप्त केला.","अंतिम धावसंख्या म्हणजे २१ ते २० असा एक गुणांचा विजय होता, ज्यामुळे All Blacks १५ गेमजिंकण्याचीमालिका संपुष्टात आली.",Marathi,0.20,0.64
3,"The final score was a one-point victory, 21 to 20, ending the All Blacks' 15 game winning streak.","अखेरच्या धावांनी 21 ते 20 असा एक गुण राखून विजय मिळविला, ज्याने ऑल ब्लॅकचा 15 गेम जिंकणारा वेग समाप्त केला.","अंतिम धावसंख्या म्हणजे २१ ते २० असा एक गुणांचाविजय होता, ज्यामुळे All Blacks १५ गेमजिंकण्याची मालिका संपुष्टात आली.",Marathi,0.64,0.60
4,"The final score was a one-point victory, 21 to 20, ending the All Blacks' 15 game winning streak.","अखेरच्या धावांनी 21 ते 20 असा एक गुण राखून विजय मिळविला, ज्याने ऑल ब्लॅकचा 15 गेम जिंकणारा वेग समाप्त केला.","अंतिम गुण एक-गुणाचा विजय होता, 21-20, सर्व ब्लॅक्सची 15सामन्यांची विजयी मालिका समाप्त झाली.",Marathi,0.92,0.92


IndicMT Test
Rows           : 2380
Columns        : 6
Missing Values : 0
Duplicate Rows : 22

Column Names
['src', 'ref', 'translation', 'language', 'mqm_norm_score', 'da_norm_score']

Data Types
src                object
ref                object
translation        object
language           object
mqm_norm_score    float64
da_norm_score     float64
dtype: object

Sample Data


,src,ref,translation,language,mqm_norm_score,da_norm_score
0,"The Chaco region was home to other groups of indigenous tribes such as the Guaycurú and Payaguá, who survived by hunting, gathering and fishing.","ਚਾਕੋ ਖੇਤਰ ਦੇਸੀ ਕਬੀਲਿਆਂ ਦੇ ਹੋਰ ਸਮੂਹਾਂ ਜਿਵੇਂ ਗਾਇਕੂਰੁ ਅਤੇ ਪਾਇਗੂਆ ਦਾ ਘਰ ਸੀ, ਜੋ ਸ਼ਿਕਾਰ, ਇਕੱਠੇ ਰਹਿਣ ਅਤੇ ਫਿਸ਼ਿੰਗ ਦੁਆਰਾ ਬਚ ਗਏ ਸਨ।","ਚਾਕੋ ਖੇਤਰ ਸਵਦੇਸ਼ੀ ਕਬੀਲਿਆਂ ਦੇ ਹੋਰ ਸਮੂਹਾਂ ਦਾ ਘਰ ਸੀ ਜਿਵੇਂ ਕਿ ਗੁਆਏਕੁਰੂ ਅਤੇ ਪਯਾਗੁਆ, ਜੋ ਸ਼ਿਕਾਰ, ਇਕੱਠੇ ਕਰਨ ਅਤੇ ਮੱਛੀਆਂ ਫੜ ਕੇ ਬਚੇ ਸਨ।",Punjabi,0.84,0.40
1,"For instance, they didn't have corn, nor tomatoes, nor potatoes, nor cocoa, and no ancient Roman ever tasted a turkey.","ਮਿਸਾਲ ਦੇ ਤੌਰ ‘ਤੇ, ਉਨ੍ਹਾਂ ਕੋਲ ਨਾ ਮੱਕੀ, ਨਾ ਟਮਾਟਰ, ਨਾ ਆਲੂ, ਨਾ ਕੋਕੋਆ ਸੀ ਅਤੇ ਨਾ ਹੀ ਕਿਸੇ ਪ੍ਰਾਚੀਨ ਰੋਮਨ ਨੇ ਕਦੇ ਟਰਕੀ ਦਾ ਸੁਆਦ ਦੇਖਿਆ ਸੀ।","ਉਦਾਹਰਣ ਵਜੋਂ, ਉਨ੍ਹਾਂ ਕੋਲ ਮੱਕੀ, ਨਾ ਟਮਾਟਰ, ਨਾ ਆਲੂ, ਨਾ ਕੋਕੋ ਸੀ ਅਤੇ ਕਿਸੇ ਵੀ ਪ੍ਰਾਚੀਨ ਰੋਮਨ ਨੇ ਕਦੇ ਟਰਕੀ ਦਾ ਸੁਆਦ ਨਹੀਂ ਚੱਖਿਆ ਸੀ।",Punjabi,0.84,0.60
2,"Cochamó Valley - Chile's premier climbing destination, known as the Yosemite of South America, with a variety of granite big walls and crags.","ਕੋਕੈਮੋ ਘਾਟੀ - ਚਿੱਲੀ ਦਾ ਪ੍ਰਮੁੱਖ ਚੜ੍ਹਾਈ ਵਾਲਾ ਭਾਗ, ਜੋ ਦੱਖਣੀ ਅਮਰੀਕਾ ਦੀਆਂ ਸਖ਼ਤ ਵੱਡੀਆਂ ਦੀਵਾਰਾਂ ਦੀਆਂ ਕਿਸਮਾਂ ਅਤੇ ਕ੍ਰੇਗ ਨਾਲ ਯੋਸੇਮਾਈਟ ਦੇ ਰੂਪ ਵਿੱਚ ਜਾਣਿਆ ਜਾਂਦਾ ਹੈ।","ਕੋਚਾਮੋ ਵੈਲੀ - ਚਿਲੀ ਦੀ ਪ੍ਰਮੁੱਖ ਚੜ੍ਹਾਈ ਦੀ ਮੰਜ਼ਿਲ, ਜਿਸ ਨੂੰ ਦੱਖਣੀ ਅਮਰੀਕਾ ਦੇ ਯੋਸੇਮਾਈਟ ਵਜੋਂ ਜਾਣਿਆ ਜਾਂਦਾ ਹੈ, ਜਿਸ ਦੀਆਂ ਕਈ ਤਰ੍ਹਾਂ ਦੀਆਂ ਗ੍ਰੇਨਾਈਟ ਵੱਡੀਆਂ ਕੰਧਾਂ ਅਤੇ ਕ੍ਰੈਗ ਹਨ।",Punjabi,0.84,0.40
3,During these periods fierce warfare took place between many nobles fighting for the throne.,ਇਹਨਾਂ ਸਮਿਆਂ ਦੌਰਾਨ ਤਖ਼ਤ ਵਾਸਤੇ ਲੜ ਰਹੇ ਕਈ ਖ਼ਾਨਦਾਨਾਂ ਵਿਚਾਲੇ ਭਿਆਨਕ ਯੁੱਧ ਹੋਇਆ।,ਇਨ੍ਹਾਂ ਸਮਿਆਂ ਦੌਰਾਨ ਸਿੰਘਾਸਨ ਲਈ ਲੜ ਰਹੇ ਬਹੁਤ ਸਾਰੇ ਕੁਲੀਨਾਂ ਵਿਚਕਾਰ ਭਿਆਨਕ ਯੁੱਧ ਹੋਇਆ।,Punjabi,0.92,0.88
4,Travellers are strongly advised to be aware of any risk of severe weather affecting their area as they may affect any travel plans.,ਮੁਸਾਫ਼ਰਾਂ ਨੂੰ ਉਹਨਾਂ ਦੇ ਖੇਤਰ ਦੇ ਅਸਰ ਕਰਨ ਵਾਲੇ ਭਿਆਨਕ ਮੌਸਮ ਤੋਂ ਹੋਣ ਵਾਲੇ ਕਿਸੇ ਵੀ ਤਰ੍ਹਾਂ ਦੇ ਖਤਰੇ ਤੋਂ ਸਾਵਧਾਨ ਰਹਿਣ ਦੀ ਸਲਾਹ ਦਿੱਤੀ ਜਾਂਦੀ ਹੈ ਕਿਉਂਕਿ ਉਹ ਉਹਨਾਂ ਦੀ ਯਾਤਰਾ ਦੀ ਯੋਜਨਾ 'ਤੇ ਅਸਰ ਪਾ ਸਕਦੇ ਹਨ।,ਯਾਤਰੀਆਂ ਨੂੰ ਜ਼ੋਰਦਾਰ ਸਲਾਹ ਦਿੱਤੀ ਜਾਂਦੀ ਹੈ ਕਿ ਉਹ ਆਪਣੇ ਖੇਤਰ ਨੂੰ ਪ੍ਰਭਾਵਿਤ ਕਰਨ ਵਾਲੇ ਕਿਸੇ ਵੀ ਖਤਰਨਾਕ ਮੌਸਮ ਦੇ ਜੋਖਮ ਤੋਂ ਜਾਣੂ ਹੋਣ ਕਿਉਂਕਿ ਉਹ ਕਿਸੇ ਵੀ ਯਾਤਰਾ ਯੋਜਨਾ ਨੂੰ ਪ੍ਰਭਾਵਤ ਕਰ ਸਕਦੇ ਹਨ।,Punjabi,0.88,0.80


{'Dataset': 'IndicMT Test',
 'Rows': 2380,
 'Columns': 6,
 'Missing Values': 0,
 'Duplicate Rows': 22}

In [ ]:
print(train.columns.tolist())

['src', 'ref', 'translation', 'language', 'mqm_norm_score', 'da_norm_score']


In [ ]:
def standardize_indicmt(df):

    standardized = df.rename(columns={
        "src": "source",
        "ref": "reference"
    })

    standardized["dataset"] = "IndicMT-Eval"

    return standardized

In [ ]:
train_std = standardize_indicmt(train)
train_std.head()

,source,reference,translation,language,mqm_norm_score,da_norm_score,dataset
0,NaN,NaN,NaN,Gujarati,NaN,NaN,IndicMT-Eval
1,. Scientists say this animal's plumage was chestnut-brown on top with a pale or carotenoid-colored underside.,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથીકથ્થાઈ-બદામી રંગના અને અંદરની બાજુથી આછા અથવા પીળા રંગના હતા.,એમ.વૈજ્ઞાનિકોનું કહેવું છે કે આ પ્રાણીના પીંછાં પીળા અથવા કેરોટિનોઇડ રંગના નીચેના ભાગ સાથે ટોચ પર ચેસ્ટનટ-બ્રાઉન રંગના હતા.,Gujarati,0.90,0.8,IndicMT-Eval
2,. Scientists say this animal's plumage was chestnut-brown on top with a pale or carotenoid-colored underside.,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથીકથ્થાઈ-બદામી રંગના અને અંદરની બાજુથી આછા અથવા પીળા રંગના હતા.,. વૈજ્entistsાનિકોકહે છે કે આ પ્રાણીનું પ્લgeમજ નિસ્તેજ અથવા કેરોટિનોઇડ રંગના અન્ડરસાઇડ સાથે ટોચ પર ચેસ્ટનટ-બ્રાઉન હતું.,Gujarati,0.60,0.6,IndicMT-Eval
3,. Scientists say this animal's plumage was chestnut-brown on top with a pale or carotenoid-colored underside.,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથીકથ્થાઈ-બદામી રંગના અને અંદરની બાજુથી આછા અથવા પીળા રંગના હતા.,. વૈજ્ઞાનિકોનું કહેવું છે કે આ પ્રાણીનું પ્લમેજ ટોચ પર ચેસ્ટનટ-બ્રાઉન હતું અને તેની નીચે ફિક્કો અથવા કેરોટીનોઇડ રંગનોહતો.,Gujarati,0.88,0.8,IndicMT-Eval
4,. Scientists say this animal's plumage was chestnut-brown on top with a pale orcarotenoid-colored underside.,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથીકથ્થાઈ-બદામી રંગના અને અંદરની બાજુથી આછા અથવા પીળા રંગના હતા.,વૈજ્ઞાનિકોનું કહેવું છે કે આ પ્રાણીની પલમેજ ઉપરના ભાગમાં ચૈસ્ટનટ-બ્રાઉન હતી અને તેની નીચેના ભાગમાં કેરોટેનોઈડનો રંગ હતો.,Gujarati,0.86,0.8,IndicMT-Eval


In [ ]:
INDIC_OUTPUT = PROJECT_ROOT / "processed" / "indicmt"
INDIC_OUTPUT.mkdir(parents=True, exist_ok=True)

validation_std = standardize_indicmt(validation)
test_std = standardize_indicmt(test)

train_std.to_csv(INDIC_OUTPUT / "train_standardized.csv", index=False)
validation_std.to_csv(INDIC_OUTPUT / "validation_standardized.csv", index=False)
test_std.to_csv(INDIC_OUTPUT / "test_standardized.csv", index=False)

print("✅ IndicMT standardized datasets saved successfully.")

✅ IndicMT standardized datasets saved successfully.


In [ ]:
indic_summary = pd.DataFrame([
    {
        "Dataset": "Train",
        "Rows": len(train_std),
        "Columns": len(train_std.columns),
        "Missing Values": train_std.isnull().sum().sum(),
        "Duplicate Rows": train_std.duplicated().sum(),
    },
    {
        "Dataset": "Validation",
        "Rows": len(validation_std),
        "Columns": len(validation_std.columns),
        "Missing Values": validation_std.isnull().sum().sum(),
        "Duplicate Rows": validation_std.duplicated().sum(),
    },
    {
        "Dataset": "Test",
        "Rows": len(test_std),
        "Columns": len(test_std.columns),
        "Missing Values": test_std.isnull().sum().sum(),
        "Duplicate Rows": test_std.duplicated().sum(),
    }
])

indic_summary

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,Train,4997,7,5,25
1,Validation,1000,7,0,7
2,Test,2380,7,0,22


In [ ]:
train_std.isnull().sum()

,0
source,1
reference,1
translation,1
language,0
mqm_norm_score,1
da_norm_score,1
dataset,0


In [ ]:
train_std[train_std.isnull().any(axis=1)]

,source,reference,translation,language,mqm_norm_score,da_norm_score,dataset
0,NaN,NaN,NaN,Gujarati,NaN,NaN,IndicMT-Eval


In [ ]:
train_std = train_std.dropna().reset_index(drop=True)

In [ ]:
train_std.isnull().sum()

,0
source,0
reference,0
translation,0
language,0
mqm_norm_score,0
da_norm_score,0
dataset,0


In [ ]:
duplicates = train_std[train_std.duplicated(keep=False)]

print("Duplicate rows:", len(duplicates))

duplicates.head(10)

Duplicate rows: 44


,source,reference,translation,language,mqm_norm_score,da_norm_score,dataset
297,Fellow wrestlers also paid tribute to Luna.,સાથી કુસ્તીબાજોએ પણ લુનાને શ્રદ્ધાંજલિ આપી હતી.,સાથી કુસ્તીબાજોએ પણ લુનાને શ્રદ્ધાંજલિ આપી હતી.,Gujarati,1.0,1.0,IndicMT-Eval
300,Fellow wrestlers also paid tribute to Luna.,સાથી કુસ્તીબાજોએ પણ લુનાને શ્રદ્ધાંજલિ આપી હતી.,સાથી કુસ્તીબાજોએ પણ લુનાને શ્રદ્ધાંજલિ આપી હતી.,Gujarati,1.0,1.0,IndicMT-Eval
302,Fellow wrestlers also paid tribute to Luna.,સાથી કુસ્તીબાજોએ પણ લુનાને શ્રદ્ધાંજલિ આપી હતી.,સાથી કુસ્તીબાજોએ પણ લુનાને શ્રદ્ધાંજલિ આપી હતી.,Gujarati,1.0,1.0,IndicMT-Eval
1297,Fellow wrestlers also paid tribute to Luna.,സഹ ഗുസ്തിക്കാരും ലൂണയ്ക്ക് ആദരാഞ്ജലികൾ അർപ്പിച്ചു.,സഹ ഗുസ്തിക്കാരും ലൂണയ്ക്ക് ആദരാഞ്ജലി അർപ്പിച്ചു.,Malayalam,1.0,1.0,IndicMT-Eval
1299,Fellow wrestlers also paid tribute to Luna.,സഹ ഗുസ്തിക്കാരും ലൂണയ്ക്ക് ആദരാഞ്ജലികൾ അർപ്പിച്ചു.,സഹ ഗുസ്തിക്കാരും ലൂണയ്ക്ക് ആദരാഞ്ജലി അർപ്പിച്ചു.,Malayalam,1.0,1.0,IndicMT-Eval
1695,"Naturalists and philosophers focused on classical texts and, in particular, on the Bible in Latin.",പ്രകൃതിവാദികളും തത്ത്വചിന്തകരും ക്ലാസിക്കൽ വാക്യങ്ങളിലും പ്രത്യേകിച്ച് ലാറ്റിൻ ഭാഷയിലുള്ള ബൈബിളിലും ശ്രദ്ധ കേന്ദ്രീകരിച്ചു.,പ്രകൃതിശാസ്ത്രജ്ഞരും തത്ത്വചിന്തകരും ക്ലാസിക്കൽ ഗ്രന്ഥങ്ങളിലും പ്രത്യേകിച്ച് ലാറ്റിൻ ഭാഷയിലുള്ള ബൈബിളിലും ശ്രദ്ധ കേന്ദ്രീകരിച്ചു.,Malayalam,1.0,1.0,IndicMT-Eval
1698,"Naturalists and philosophers focused on classical texts and, in particular, on the Bible in Latin.",പ്രകൃതിവാദികളും തത്ത്വചിന്തകരും ക്ലാസിക്കൽ വാക്യങ്ങളിലും പ്രത്യേകിച്ച് ലാറ്റിൻ ഭാഷയിലുള്ള ബൈബിളിലും ശ്രദ്ധ കേന്ദ്രീകരിച്ചു.,പ്രകൃതിശാസ്ത്രജ്ഞരും തത്ത്വചിന്തകരും ക്ലാസിക്കൽ ഗ്രന്ഥങ്ങളിലും പ്രത്യേകിച്ച് ലാറ്റിൻ ഭാഷയിലുള്ള ബൈബിളിലും ശ്രദ്ധ കേന്ദ്രീകരിച്ചു.,Malayalam,1.0,1.0,IndicMT-Eval
1737,One bomb exploded outside the governor general's office.,ഗവർണർ ജനറലിന്റെ ഓഫീസിന് പുറത്ത് ഒരു ബോംബ് പൊട്ടിത്തെറിച്ചു.,ഗവർണർ ജനറലിന്റെ ഓഫീസിന് പുറത്ത് ഒരു ബോംബ് പൊട്ടിത്തെറിച്ചു.,Malayalam,1.0,1.0,IndicMT-Eval
1740,One bomb exploded outside the governor general's office.,ഗവർണർ ജനറലിന്റെ ഓഫീസിന് പുറത്ത് ഒരു ബോംബ് പൊട്ടിത്തെറിച്ചു.,ഗവർണർ ജനറലിന്റെ ഓഫീസിന് പുറത്ത് ഒരു ബോംബ് പൊട്ടിത്തെറിച്ചു.,Malayalam,1.0,1.0,IndicMT-Eval
2239,"Curtis Cooper, a mathematician and computer science professor at the University of Central Missouri, has discovered the largest known prime number to date on January 25.","कर्टिस कूपर, सेंट्रल मिसौरी विश्वविद्यालय में एक गणितज्ञ और कंप्यूटर विज्ञान के प्रोफेसर ने 25 जनवरी को अब तक की सबसे बड़ी ज्ञात प्राइम संख्या की खोज की है।",सेंट्रल मिसौरी विश्वविद्यालय में गणितज्ञ और कंप्यूटर विज्ञान के प्रोफेसर कर्टिस कूपर ने 25 जनवरी को अब तक की सबसे बड़ी ज्ञात अभाज्य संख्या की खोज की है।,Hindi,1.0,1.0,IndicMT-Eval


In [ ]:
train_std = train_std.drop_duplicates().reset_index(drop=True)
validation_std = validation_std.drop_duplicates().reset_index(drop=True)
test_std = test_std.drop_duplicates().reset_index(drop=True)

In [ ]:
print("Train:", train_std.duplicated().sum())
print("Validation:", validation_std.duplicated().sum())
print("Test:", test_std.duplicated().sum())

Train: 0
Validation: 0
Test: 0


In [ ]:
INDIC_OUTPUT = PROJECT_ROOT / "processed" / "indicmt"
INDIC_OUTPUT.mkdir(parents=True, exist_ok=True)

train_std.to_csv(INDIC_OUTPUT / "train_standardized.csv", index=False)
validation_std.to_csv(INDIC_OUTPUT / "validation_standardized.csv", index=False)
test_std.to_csv(INDIC_OUTPUT / "test_standardized.csv", index=False)

print("✅ Final cleaned and standardized IndicMT datasets saved.")

✅ Final cleaned and standardized IndicMT datasets saved.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

OUTPUT_DIR = PROJECT_ROOT / "processed" / "indicmt"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
train_std.to_csv(
    OUTPUT_DIR / "train_standardized.csv",
    index=False
)

validation_std.to_csv(
    OUTPUT_DIR / "validation_standardized.csv",
    index=False
)

test_std.to_csv(
    OUTPUT_DIR / "test_standardized.csv",
    index=False
)

print("✅ Files saved successfully.")

NameError: name 'train_std' is not defined

In [ ]:
from pathlib import Path

print(Path("/content/drive").exists())
print(Path("/content/drive/MyDrive").exists())

True
True


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

print(list((PROJECT_ROOT / "datasets").glob("*")))

[]


In [ ]:

print(list(PROJECT_ROOT.rglob("train_clean.csv")))

[]


In [ ]:
print(list(OUTPUT_DIR.glob("*")))

In [ ]:
pd.read_csv(
    OUTPUT_DIR / "train_standardized.csv"
).head()

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

INDICMT_DIR = PROJECT_ROOT / "datasets" / "IndicMT_Eval"

print("Folder exists:", INDICMT_DIR.exists())
print("Files:")
for file in INDICMT_DIR.iterdir():
    print(file.name)

Folder exists: False
Files:


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/UniMTScore/datasets/IndicMT_Eval'

## IndicMT-Eval Dataset Verification and Standardization

The IndicMT-Eval dataset was verified to ensure data integrity before use in the evaluation framework. The dataset includes the source sentence, human reference translation, machine-generated translation, language identifier, and normalized human evaluation scores (MQM and Direct Assessment).

To maintain consistency across all datasets, the columns `src` and `ref` were renamed to `source` and `reference`, while the `translation` column was retained to represent the machine translation output. An additional `dataset` column was added to identify the dataset origin.

The standardized datasets are saved for use in the lexical, syntactic, semantic, and validation modules of the UniMTScore framework.